# AI-Powered Gen Z Credit Card Product Innovation System
## Notebook 01 – Gen Z Customer Segmentation

Description

This notebook performs customer intelligence analysis using a synthetic Canadian Gen Z customer dataset. It prepares the data through feature engineering and applies unsupervised machine learning (K-Means clustering) to identify distinct customer segments. The resulting clusters are interpreted as business personas that represent different Gen Z customer groups. These personas serve as the foundation for AI-driven credit card product innovation in subsequent stages of the project.

Main Tasks
• Data preprocessing and feature engineering
• Customer clustering using K-Means
• Cluster evaluation and visualization
• Persona interpretation and naming
• Export customer persona profiles for downstream AI product generation

Output
• Customer persona profiles
• Cluster statistics
• Persona descriptions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:

# 1. Load dataset
df = pd.read_csv("genz_credit_card_trends_synthetic_3000.csv")

# Standardise column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df.drop(columns=["synthetic_persona"], inplace=True)
df.drop(columns=["recommended_card_positioning"], inplace=True)


print("Dataset shape:", df.shape)
print(df.head())

In [ ]:
print(df.columns)

In [ ]:
# ==========================================
# STEP 3: DATA QUALITY CHECKS
# ==========================================

print("Duplicate customer IDs:")
print(df["customer_id"].duplicated().sum())

print("\nTotal missing values:")
print(df.isna().sum().sum())

print("\nColumns with missing values:")
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ].to_frame("missing_values")
)

print("\nInfinite numeric values:")
numeric_data = df.select_dtypes(include=np.number)

print(
    np.isinf(numeric_data)
    .sum()
    .sum()
)

In [ ]:
# ==========================================
# STEP 4: SELECT CREDIT CARD HOLDERS
# ==========================================

df_analysis = df[
    df["has_credit_card"] == 1
].copy()

print(
    "Customers with a credit card:",
    len(df_analysis)
)

print(
    "Customers without a credit card:",
    (df["has_credit_card"] == 0).sum()
)

In [ ]:
# ==========================================
# STEP 5: BASIC DESCRIPTIVE ANALYSIS
# ==========================================

summary_columns = [
    "age",
    "annual_income",
    "estimated_credit_score",
    "credit_limit",
    "monthly_credit_spend",
    "monthly_debit_spend",
    "utilization_rate",
    "social_media_hours_per_day"
]

display(
    df_analysis[summary_columns]
    .describe()
    .round(2)
)

In [ ]:
##View Categorical distribution

categorical_overview = [
    "province",
    "urbanicity",
    "student_status",
    "employment_status",
    "credit_score_band"
]

for column in categorical_overview:
    print(f"\n{column.upper()}")
    
    display(
        df_analysis[column]
        .value_counts(dropna=False)
        .to_frame("customer_count")
    )

In [ ]:
# ==========================================
# STEP 6: ANALYSE SPENDING CATEGORY
# ==========================================

spending_columns = [
    "groceries_spend",
    "dining_coffee_spend",
    "transit_rideshare_spend",
    "gas_carshare_spend",
    "subscriptions_streaming_spend",
    "gaming_digital_spend",
    "fashion_beauty_spend",
    "fitness_wellness_spend",
    "travel_experiences_spend",
    "tech_electronics_spend",
    "education_spend",
    "creator_marketplace_spend",
    "sustainable_products_spend"
]

average_spending = (
    df_analysis[spending_columns]
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .to_frame("average_monthly_spend")
)

display(average_spending)

In [ ]:
## Plot average spending

plt.figure(figsize=(10, 7))

plt.barh(
    average_spending.index,
    average_spending["average_monthly_spend"]
)

plt.title("Average Monthly Gen Z Spending by Category")
plt.xlabel("Average Monthly Spend")
plt.ylabel("Spending Category")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# STEP 7: CREATE COMPOSITE GEN Z FEATURES
# ==========================================

# 7.1 Social media engagement
social_media_columns = [
    "tiktok_usage_intensity",
    "instagram_usage_intensity",
    "youtube_usage_intensity",
    "snapchat_usage_intensity",
    "reddit_discord_usage_intensity",
    "social_media_hours_per_day"
]

df_analysis["social_media_engagement_score"] = (
    df_analysis[social_media_columns]
    .mean(axis=1)
)


# 7.2 Social commerce behaviour
social_commerce_columns = [
    "social_search_behavior",
    "influencer_trust",
    "ugc_review_reliance",
    "short_form_video_preference",
    "creator_economy_interest",
    "social_commerce_likelihood"
]

df_analysis["social_commerce_score"] = (
    df_analysis[social_commerce_columns]
    .mean(axis=1)
)


# 7.3 Digital banking preference
digital_banking_columns = [
    "mobile_wallet_usage",
    "virtual_card_interest",
    "instant_approval_preference",
    "budgeting_tools_interest",
    "spending_alerts_interest",
    "fraud_alert_importance"
]

df_analysis["digital_banking_score"] = (
    df_analysis[digital_banking_columns]
    .mean(axis=1)
)


# 7.4 Combined digital engagement
df_analysis["digital_engagement_score"] = (
    df_analysis[
        [
            "social_media_engagement_score",
            "social_commerce_score",
            "digital_banking_score"
        ]
    ]
    .mean(axis=1)
)


# 7.5 Financial wellness
financial_wellness_columns = [
    "financial_confidence",
    "savings_priority",
    "credit_building_interest",
    "budgeting_tools_interest",
    "spending_alerts_interest"
]

df_analysis["financial_wellness_score"] = (
    df_analysis[financial_wellness_columns]
    .mean(axis=1)
)


# 7.6 Financial pressure
financial_pressure_columns = [
    "debt_concern",
    "budgeting_need",
    "price_sensitivity",
    "missed_payment_risk",
    "utilization_rate"
]

df_analysis["financial_pressure_score"] = (
    df_analysis[financial_pressure_columns]
    .mean(axis=1)
)


# 7.7 Reward preference
reward_preference_columns = [
    "cashback_interest",
    "subscription_rewards_interest",
    "transit_rewards_interest",
    "travel_rewards_interest",
    "dining_rewards_interest",
    "sustainability_rewards_interest",
    "credit_building_interest",
    "low_fee_interest"
]

df_analysis["reward_preference_score"] = (
    df_analysis[reward_preference_columns]
    .mean(axis=1)
)


# 7.8 Experience and lifestyle orientation
experience_lifestyle_columns = [
    "wellness_interest",
    "travel_intent",
    "novelty_exploration",
    "community_values",
    "status_experience_drive"
]

df_analysis["experience_lifestyle_score"] = (
    df_analysis[experience_lifestyle_columns]
    .mean(axis=1)
)


# 7.9 Sustainability Score
df_analysis["sustainability_score"] = (
    df_analysis[
        [
            "sustainability_interest",
            "sustainability_rewards_interest"
        ]
    ]
    .mean(axis=1)
)


# 7.10 Creator Economy Score
df_analysis["creator_economy_score"] = (
    df_analysis[
        [
            "creator_economy_interest",
            "side_hustle_interest",
            "social_commerce_likelihood"
        ]
    ]
    .mean(axis=1)
)


# 7.11 Credit spending share
df_analysis["credit_spend_share"] = (
    df_analysis["monthly_credit_spend"]
    /
    (
        df_analysis["monthly_credit_spend"]
        + df_analysis["monthly_debit_spend"]
        + 1
    )
)


# Everyday spending score
everyday_spending_columns = [
    "groceries_spend",
    "transit_rideshare_spend",
    "gas_carshare_spend",
    "education_spend"
]

df_analysis["everyday_spending_score"] = (
    df_analysis[everyday_spending_columns].sum(axis=1)
)


# Lifestyle spending score
lifestyle_spending_columns = [
    "dining_coffee_spend",
    "subscriptions_streaming_spend",
    "gaming_digital_spend",
    "fashion_beauty_spend",
    "fitness_wellness_spend",
    "travel_experiences_spend",
    "tech_electronics_spend",
    "creator_marketplace_spend",
    "sustainable_products_spend"
]

df_analysis["lifestyle_spending_score"] = (
    df_analysis[lifestyle_spending_columns].sum(axis=1)
)


# Lifestyle spending share
df_analysis["lifestyle_spend_share"] = (
    df_analysis["lifestyle_spending_score"]
    /
    (
        df_analysis["lifestyle_spending_score"]
        + df_analysis["everyday_spending_score"]
        + 1
    )
)


print("Step 7 completed successfully.")

In [ ]:
for col in df_analysis.columns:
    print(col)

In [ ]:
new_features = [
    "social_media_engagement_score",
    "social_commerce_score",
    "digital_banking_score",
    "digital_engagement_score",
    "financial_wellness_score",
    "financial_pressure_score",
    "reward_preference_score",
    "experience_lifestyle_score",
    "sustainability_score",
    "creator_economy_score",
    "credit_spend_share",
    "lifestyle_spend_share"
]

display(df_analysis[new_features].head())

In [ ]:
# ==========================================
# STEP 8: SELECT FINAL CLUSTERING FEATURES
# ==========================================

cluster_features = [

    # Demographics and life stage
    "age",
    "student_status",
    "employment_status",
    "annual_income",
    "urbanicity",

    # Credit behaviour
    "estimated_credit_score",
    "credit_limit",
    "utilization_rate",
    "pays_in_full_probability",
    "missed_payment_risk",

    # Composite spending scores
    "everyday_spending_score",
    "lifestyle_spending_score",

    # Digital and social behaviour
    "social_media_engagement_score",
    "social_commerce_score",
    "digital_banking_score",
    "digital_engagement_score",

    # Financial behaviour
    "financial_wellness_score",
    "financial_pressure_score",

    # Preferences and values
    "reward_preference_score",
    "experience_lifestyle_score",
    "sustainability_score",
    "creator_economy_score",

    # Spending pattern ratios
    "credit_spend_share",
    "lifestyle_spend_share"
]

# Keep only columns that actually exist
cluster_features = [
    col for col in cluster_features
    if col in df_analysis.columns
]

df_model = df_analysis[
    cluster_features
].copy()

print("Number of clustering features:", len(cluster_features))
print("Model dataset shape:", df_model.shape)

print("\nClustering features:")
for col in cluster_features:
    print(col)

In [ ]:
# ==========================================
# STEP 9: CHECK MODEL DATA
# ==========================================

df_model = df_model.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Total missing values:",
    df_model.isna().sum().sum()
)

missing_summary = (
    df_model.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ].to_frame("missing_values")
)

In [ ]:
# ==========================================
# STEP 10: IDENTIFY DATA TYPES
# ==========================================

numeric_cols = (
    df_model
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_cols = (
    df_model
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)

In [ ]:
# ==========================================
# STEP 11: PREPROCESSING PIPELINE
# ==========================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_cols
    )
])

print("Preprocessing pipeline created successfully.")

In [ ]:
# ==========================================
# STEP 12: TRANSFORM MODEL DATA
# ==========================================

X_processed = preprocessor.fit_transform(
    df_model
)

print("Processed data shape:", X_processed.shape)

print(
    "Contains missing values:",
    np.isnan(X_processed).any()
)

print(
    "Contains infinite values:",
    np.isinf(X_processed).any()
)

In [ ]:
# ==========================================
# STEP 13: TEST NUMBER OF CLUSTERS
# ==========================================

cluster_results = []

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(
        X_processed
    )

    silhouette = silhouette_score(
        X_processed,
        labels
    )

    cluster_results.append({
        "number_of_clusters": k,
        "silhouette_score": silhouette,
        "inertia": kmeans.inertia_
    })


cluster_results_df = pd.DataFrame(
    cluster_results
)

display(
    cluster_results_df.round(4)
)

In [ ]:
# ==========================================
# STEP 14: SILHOUETTE SCORE PLOT
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    cluster_results_df["number_of_clusters"],
    cluster_results_df["silhouette_score"],
    marker="o"
)

plt.title(
    "Silhouette Score by Number of Gen Z Clusters"
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")

plt.xticks(
    cluster_results_df["number_of_clusters"]
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# STEP 15: ELBOW METHOD
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    cluster_results_df["number_of_clusters"],
    cluster_results_df["inertia"],
    marker="o"
)

plt.title(
    "Elbow Method for Gen Z Clustering"
)

plt.xlabel("Number of Clusters")
plt.ylabel("K-Means Inertia")

plt.xticks(
    cluster_results_df["number_of_clusters"]
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# STEP 16: IDENTIFY BEST K
# ==========================================

best_row = cluster_results_df.loc[
    cluster_results_df[
        "silhouette_score"
    ].idxmax()
]

best_k = int(
    best_row["number_of_clusters"]
)

best_score = float(
    best_row["silhouette_score"]
)

print(
    "Statistically recommended number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)

In [ ]:
four_cluster_score = cluster_results_df.loc[
    cluster_results_df[
        "number_of_clusters"
    ] == 4,
    "silhouette_score"
].iloc[0]

print(
    "Silhouette score for four clusters:",
    round(four_cluster_score, 4)
)

In [ ]:
display(cluster_results_df)

Statistically, the data naturally separates into 2 clusters.

However, your project is not simply to find the statistically optimal number of clusters. Your objective is to:

identify distinct Gen Z customer segments,
build personas,
recommend credit card products.

In customer segmentation, it's common to balance statistical fit with business usefulness.

My recommendation

I would keep 4 clusters.

Here's why:

The drop from 3 clusters (0.0763) to 4 clusters (0.0689) is only 0.0074, which is quite small.
Four personas are much more actionable for a bank than two broad groups.
Four segments let you differentiate customers by financial behaviour, lifestyle, digital engagement, and rewards preferences.

In your report, you can write something like:

Although a two-cluster solution achieved the highest silhouette score (0.114), a four-cluster solution was selected because it provided greater business interpretability and enabled the development of four distinct Gen Z personas for targeted credit card recommendations. The reduction in silhouette score was modest and was considered an acceptable trade-off for richer segmentation.

That's a strong, defensible justification.

In [ ]:
# ==========================================
# STEP 17: FINAL K-MEANS MODEL
# ==========================================

final_k = 4

kmeans = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=20
)

df_analysis["cluster"] = kmeans.fit_predict(X_processed)

print("Final clustering completed.")
print(df_analysis["cluster"].value_counts().sort_index())

In [ ]:
cluster_profile = (
    df_analysis
    .groupby("cluster")
    .mean(numeric_only=True)
)

cluster_profile

In [ ]:
# ==========================================
# STEP 18: CHECK CLUSTER SIXES
# ==========================================

cluster_sizes = (
    df_analysis["cluster"]
    .value_counts()
    .sort_index()
    .to_frame("Customers")
)

cluster_sizes["Percentage"] = (
    cluster_sizes["Customers"] /
    len(df_analysis) * 100
).round(1)

display(cluster_sizes)

The final four-cluster solution produced well-balanced customer segments, with each cluster representing approximately one-quarter of the synthetic Gen Z population. This indicates that the segmentation captures distinct behavioural groups without creating disproportionately small or overly dominant segments

In [ ]:
# ==========================================
# STEP 19: CLUSTER PROFILE
# ==========================================

profile_columns = [

    # Demographics
    "age",
    "annual_income",

    # Credit
    "estimated_credit_score",
    "credit_limit",
    "utilization_rate",
    "pays_in_full_probability",

    # Spending
    "everyday_spending_score",
    "lifestyle_spending_score",

    # Behaviour
    "social_media_engagement_score",
    "social_commerce_score",
    "digital_banking_score",
    "digital_engagement_score",

    # Financial
    "financial_wellness_score",
    "financial_pressure_score",

    # Preferences
    "reward_preference_score",
    "experience_lifestyle_score",
    "sustainability_score",
    "creator_economy_score",

    # Ratios
    "credit_spend_share",
    "lifestyle_spend_share"
]

cluster_profile = (
    df_analysis
    .groupby("cluster")[profile_columns]
    .mean()
    .round(2)
)

# Display all columns
pd.set_option("display.max_columns", None)

display(cluster_profile)

In [ ]:
pd.set_option("display.max_columns", None)

display(cluster_profile)

display(cluster_profile.T)

In [ ]:
# ==========================================
# STEP 20: CLUSTER INDEX
# Compare clusters against Gen Z average
# ==========================================

overall_average = df_analysis[profile_columns].mean()

cluster_index = (
    cluster_profile
    .div(overall_average)
    * 100
).round(1)

display(cluster_index)

In [ ]:
# ==========================================
# STEP 21: TOP CHARACTERISTICS OF EACH CLUSTER
# ==========================================

for cluster in cluster_index.index:

    print("=" * 70)
    print(f"CLUSTER {cluster}")
    print("=" * 70)

    display(
        cluster_index.loc[cluster]
        .sort_values(ascending=False)
        .head(10)
    )

In [ ]:
# ==========================================
# STEP 22: LOWEST CHARACTERISTICS
# ==========================================

for cluster in cluster_index.index:

    print("=" * 70)
    print(f"CLUSTER {cluster}")
    print("=" * 70)

    display(
        cluster_index.loc[cluster]
        .sort_values()
        .head(10)
    )

In [ ]:
# ==========================================
# STEP 23: DOMINANT CATEGORICAL CHARACTERISTICS
# ==========================================

categorical_profile = {}

categorical_columns = [
    "student_status",
    "employment_status",
    "urbanicity",
    "environics_segment",
    "psychographic_quadrant"
]

for col in categorical_columns:

    categorical_profile[col] = (
        df_analysis
        .groupby("cluster")[col]
        .agg(lambda x: x.value_counts().index[0])
    )

for col in categorical_profile:

    print("\n")
    print("=" * 60)
    print(col)
    print("=" * 60)

    display(categorical_profile[col])

In [ ]:
# ==========================================
# STEP 24: ASSIGN PERSONA NAMES
# ==========================================

cluster_names = {
    0: "Premium Lifestyle Customer",
    1: "Digital Lifestyle Customer",
    2: "Credit Builder",
    3: "Financial Growth Customer"
    
}

df_analysis["persona"] = (
    df_analysis["cluster"]
    .map(cluster_names)
)

display(
    df_analysis[
        [
            "cluster",
            "persona"
        ]
    ].drop_duplicates()
)

In [ ]:
# ==========================================
# STEP 25: PERSONA SUMMARY
# ==========================================

persona_summary = (
    df_analysis
    .groupby("persona")[
        [
            "annual_income",
            "estimated_credit_score",
            "credit_limit",
            "everyday_spending_score",
            "lifestyle_spending_score",
            "digital_engagement_score",
            "financial_wellness_score",
            "experience_lifestyle_score"
        ]
    ]
    .mean()
    .round(2)
)

display(persona_summary)

In [ ]:
# ==========================================
# STEP 26: CREDIT CARD RECOMMENDATIONS
# ==========================================

card_recommendations = {

    "Financially Savvy Professional":
        "Premium cashback card with digital banking tools and travel rewards.",

    "Lifestyle & Experience Student":
        "Student card with dining, streaming and transit rewards plus budgeting tools.",

    "Budget-Conscious Credit Builder":
        "No-fee credit builder card with cashback and credit score tracking.",

    "Emerging Lifestyle Achiever":
        "Lifestyle rewards card with fitness, travel and subscription benefits."

}

df_analysis["recommended_credit_card"] = (
    df_analysis["persona"]
    .map(card_recommendations)
)

In [ ]:
# ==========================================
# STEP 27: PCA VISUALIZATION
# ==========================================

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_processed)

plt.figure(figsize=(10,7))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df_analysis["cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Gen Z Customer Segments")

plt.colorbar(scatter,label="Cluster")

plt.show()

In [ ]:
# ==========================================
# STEP 28: SAVE DATASET
# ==========================================

df_analysis.to_csv(
    "genz_customer_segments_final.csv",
    index=False
)

print("Dataset saved successfully.")

Cluster	Persona	Key Characteristics	Product Opportunity
0	Affluent Lifestyle Explorers	Highest income ($49.6k), highest everyday and lifestyle spending, high credit score (727), highest credit utilization	Premium lifestyle/travel rewards
1	Digital Creators	Low income ($23.8k), highest social commerce (0.66), highest digital banking (0.81), highest creator economy (0.58), highest experience score (0.55)	Creator economy and digital-first card
2	Financially Responsible Young Professionals	High income ($36.1k), highest credit score (746), highest credit limit, highest pay-in-full probability (0.72), strongest financial wellness	Premium cashback / wealth-building card
3	Budget-Conscious Students	Youngest (20.9), lowest income ($21.6k), lowest credit score (677), highest debt concern and reward preference	Student credit-building card
Why this segmentation works well

The clusters are clearly differentiated on the variables that matter most:

Income ranges from $21.6k to $49.6k.
Credit score ranges from 677 to 746.
Credit limit ranges from $4,141 to $7,519.
Lifestyle spending ranges from $526 to $1,516.
Creator economy score is highest in Cluster 1.
Financial wellness is highest in Cluster 2.
Financial pressure is highest in Cluster 3.

These are meaningful business differences that can support different product strategies.

One interesting observation

The digital behavior variables are relatively close across clusters:

Social media engagement: 1.13–1.34
Digital engagement: 0.83–0.87
Reward preference: 0.65–0.66

This actually makes sense because all of your customers are Gen Z. Digital engagement is a shared characteristic of the population, so it won't strongly separate clusters. Instead, the segmentation is driven more by financial behavior, income, spending, and credit characteristics, which is exactly what a bank would want for product design.

I would describe the clusters in your report like this:
Affluent Lifestyle Explorers – High-income, high-spending customers interested in premium lifestyle benefits.
Digital Creators – Digitally engaged customers with strong creator-economy and social-commerce behavior despite lower income.
Financially Responsible Young Professionals – Customers with excellent credit management, high financial wellness, and strong repayment behavior.
Budget-Conscious Students – Younger customers with limited income and greater financial pressure who would benefit from low-fee, credit-building products.

This set of personas is easy to explain, well aligned with your business objective (designing Gen Z credit card products), and should be compelling in a capstone presentation.